# CS781 Phase 2 — Legal NER (Model 1: attention decoder, `ours_hybrid*`)
Starting point: our own Phase-1 legal LM (no external weights). Two experiments in this notebook:
1. **Full fine-tuning** of the Phase-1 backbone + a new bidirectional head (teacher)
2. **Knowledge distillation** into a half-depth student initialised from the same Phase-1 weights

Both produce a strict character-span `prediction.json` and a zip for Codabench.

## 1. Setup and imports

In [1]:
import os
import sys
import json
import glob
import time
import math
import pickle
import random
import zipfile
import subprocess

import numpy as np


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args])


try:
    import regex
except ImportError:
    pip_install("regex")
    import regex

gpu_check = "import jax, sys; sys.exit(jax.devices()[0].platform not in ('gpu', 'cuda'))"
if subprocess.run([sys.executable, "-c", gpu_check]).returncode != 0:
    pip_install("-U", "jax[cuda12]")

import jax
import jax.numpy as jnp
from jax import lax
from huggingface_hub import hf_hub_download, login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF"))
except Exception:
    pass

T0 = time.time()
random.seed(0)
np.random.seed(0)
tmap = jax.tree_util.tree_map
print("devices:", jax.devices())

devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 2. Hyperparameters and time budget

In [2]:
WINDOW = 512
TRAIN_STRIDE = 384
INFER_STRIDE = 256
BATCH = 8
O_WEIGHT = 0.25
KD_TEMP = 2.0
KD_ALPHA = 0.5
MAX_EPOCHS = 8
PEAK_LR = 5e-5
HEAD_LR_MULT = 10.0
WARMUP_STEPS = 100
BIG = 1 << 30
MAX_SECONDS = 7.5 * 3600
FULL_FT_MINUTES = 200
KD_MINUTES = 150


def budget(minutes):
    return min(minutes * 60, MAX_SECONDS - (time.time() - T0) - 2400)

## 3. Load the Phase-1 checkpoint and tokenizer

In [3]:
CK_DIR = None

found = glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)
print("checkpoints found:", found)
CK_DIR = CK_DIR or os.path.dirname(found[0])

ck = pickle.load(open(f"{CK_DIR}/final_params.pkl", "rb"))
P0 = tmap(jnp.asarray, ck["params"])
cfg = ck["cfg"]
print("variant:", ck["variant"], cfg)
assert cfg["norm"] == "rms" and cfg["mlp"] == "swiglu" and cfg["pos"] == "rope" and cfg["qknorm"]

tk = json.load(open(f"{CK_DIR}/bpe_legal.json"))

V, D = P0["emb"].shape
L = P0["layers"]["wq"].shape[0]
HD = 64
H = P0["layers"]["wq"].shape[2] // HD
KV = cfg["kvh"]
R = H // KV
WIN = np.array([256 if (cfg["hybrid"] and i % 3 != 2) else BIG for i in range(L)], np.int32)
print(f"vocab {V}, dim {D}, layers {L}, heads {H}, kv heads {KV}, local windows {sorted(set(WIN.tolist()))}")

checkpoints found: ['/kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl']
variant: ours_hybrid_gate {'norm': 'rms', 'mlp': 'swiglu', 'pos': 'rope', 'kvh': 4, 'qknorm': True, 'hybrid': True, 'gate': True, 'ff': 2048}
vocab 16384, dim 768, layers 12, heads 12, kv heads 4, local windows [256, 1073741824]


## 4. Offset-preserving tokenizer (byte-level BPE from Phase 1)

In [4]:
PAT = regex.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]|\s+""")

byte_list = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
char_list = byte_list[:]
extra = 0
for b in range(256):
    if b not in byte_list:
        byte_list.append(b)
        char_list.append(256 + extra)
        extra += 1
B2U = {b: chr(c) for b, c in zip(byte_list, char_list)}

stoi = {s: i for i, s in enumerate(tk["itos"])}
rank = {tuple(m): i for i, m in enumerate(tk["merges"])}
bpe_cache = {}


def bpe(word):
    if word not in bpe_cache:
        syms = [B2U[b] for b in word.encode()]
        while len(syms) > 1:
            best_rank, best_i = min((rank.get(p, BIG), i) for i, p in enumerate(zip(syms, syms[1:])))
            if best_rank == BIG:
                break
            syms[best_i:best_i + 2] = [syms[best_i] + syms[best_i + 1]]
        bpe_cache[word] = syms
    return bpe_cache[word]


def normalise(text):
    out, char_map = [], []
    for m in regex.finditer(r"\S+|\s+", text):
        if m.group()[0].isspace():
            if out and m.end() < len(text):
                out.append(" ")
                char_map.append(m.start())
        else:
            out.append(m.group())
            char_map.extend(range(m.start(), m.end()))
    return "".join(out), np.array(char_map)


def tokenize(text):
    norm, char_map = normalise(text)
    ids, starts_, ends_ = [], [], []
    for m in PAT.finditer(norm):
        word = m.group()
        byte_to_char = np.repeat(np.arange(len(word)), [len(c.encode()) for c in word])
        pos = 0
        for sym in bpe(word):
            nxt = pos + len(sym)
            a = m.start() + byte_to_char[pos]
            e = m.start() + byte_to_char[nxt - 1] + 1
            pos = nxt
            while a < e and norm[a] == " ":
                a += 1
            ids.append(stoi[sym])
            if a < e:
                starts_.append(char_map[a])
                ends_.append(char_map[e - 1] + 1)
            else:
                starts_.append(char_map[min(a, len(char_map) - 1)])
                ends_.append(starts_[-1])
    return np.array(ids, np.int32), np.array(starts_), np.array(ends_)

## 5. Load Phase-2 data (folds 1 and 2 for training, fold 3 for test)

In [5]:
def hf_file(name):
    return hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{name}", repo_type="dataset", local_dir="/tmp/hf")


texts, gold = {}, {}
for fold in (1, 2):
    texts.update(json.load(open(hf_file(f"Train/fold{fold}-text.json"))))
    gold.update(json.load(open(hf_file(f"Train/fold{fold}-gold.json"))))
test_texts = json.load(open(hf_file("Test/fold3-text.json")))
if isinstance(gold, list):
    gold = {str(x["id"]): x for x in gold}


def parse_entities(value):
    if isinstance(value, dict):
        value = value.get("entities", value.get("annotations", []))
    out = []
    for e in value:
        if isinstance(e, dict):
            start_keys = ("start", "start_char", "start_offset", "begin")
            end_keys = ("end", "end_char", "end_offset")
            label_keys = ("label", "type", "entity", "tag")
            s, t, l = [next(e[k] for k in keys if k in e) for keys in (start_keys, end_keys, label_keys)]
            x = e.get("text")
        else:
            s, t, l = e[0], e[1], e[2]
            x = e[3] if len(e) > 3 else None
        out.append((int(s), int(t), l, x))
    return out


raw_gold = {d: parse_entities(gold[d]) for d in texts if d in gold}

checks = [
    (texts[d][s:e] == x, texts[d][s:e + 1] == x)
    for d in raw_gold for s, e, l, x in raw_gold[d] if x
]
INC = bool(checks) and sum(c[1] for c in checks) > sum(c[0] for c in checks)
print("end offset is inclusive:", INC)

LAB = sorted({l for d in raw_gold for _, _, l, _ in raw_gold[d]})
K = len(LAB)
K2 = 2 * K + 1
print("labels:", LAB)

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

end offset is inclusive: False
labels: ['A.COUNSEL', 'APP', 'AUTH', 'CASENO', 'COURT', 'DATE', 'JUDGE', 'PREC', 'R.COUNSEL', 'RESP', 'STAT', 'WIT']


## 6. Token-level BIO labels, recoverability check and train/validation split

In [6]:
DOC = {}
GOLD = {}
recoverable, total_ents = 0, 0

for d in raw_gold:
    ids, A, B = tokenize(texts[d])
    y = np.where(B > A, 0, -1)
    for s, e, l, _ in sorted(raw_gold[d]):
        e = e + INC
        k = LAB.index(l) + 1
        ix = np.where((A >= s) & (B <= e) & (B > A))[0]
        total_ents += 1
        if len(ix):
            y[ix] = 2 * k
            y[ix[0]] = 2 * k - 1
            recoverable += int(A[ix[0]] == s and B[ix[-1]] == e)
    DOC[d] = (ids, A, B, y)
    GOLD[d] = [(s, e + INC, LAB.index(l) + 1) for s, e, l, _ in raw_gold[d]]

TEST = {d: tokenize(t) for d, t in test_texts.items()}
n_tok = sum(len(v[0]) for v in DOC.values())
print(f"train docs {len(DOC)} | test docs {len(TEST)} | tokens {n_tok} | boundary-recoverable gold entities {recoverable / total_ents:.4f}")

keys = sorted(DOC)
random.Random(0).shuffle(keys)
n_val = max(1, int(0.15 * len(keys)))
VAL_KEYS, TRAIN_KEYS = keys[:n_val], keys[n_val:]
VAL_GOLD = {d: GOLD[d] for d in VAL_KEYS}
VAL_DOCS = {d: DOC[d][:3] for d in VAL_KEYS}

counts = np.zeros(K, int)
for d in TRAIN_KEYS:
    for _, _, k in GOLD[d]:
        counts[k - 1] += 1
print("entities per label (train):", dict(zip(LAB, counts.tolist())))

train docs 70 | test docs 35 | tokens 483088 | boundary-recoverable gold entities 0.9930
entities per label (train): {'A.COUNSEL': 175, 'APP': 293, 'AUTH': 777, 'CASENO': 700, 'COURT': 916, 'DATE': 1385, 'JUDGE': 212, 'PREC': 951, 'R.COUNSEL': 153, 'RESP': 288, 'STAT': 1095, 'WIT': 143}


## 7. Sliding windows for training

In [7]:
def window_starts(n, width, stride):
    return sorted(set(list(range(0, max(n - width, 0) + 1, stride)) + [max(n - width, 0)]))


def build_windows(doc_keys, width=WINDOW, stride=TRAIN_STRIDE):
    ids_all, y_all, valid_all = [], [], []
    for d in doc_keys:
        ids, _, _, y = DOC[d]
        for s in window_starts(len(ids), width, stride):
            seg = ids[s:s + width]
            row = np.zeros(width, np.int32)
            row[:len(seg)] = seg
            lab = np.full(width, -1, np.int32)
            lab[:len(seg)] = y[s:s + width]
            ids_all.append(row)
            y_all.append(lab)
            valid_all.append(np.arange(width) < len(seg))
    return np.stack(ids_all), np.stack(y_all), np.stack(valid_all)


W_IDS, W_Y, W_VALID = build_windows(TRAIN_KEYS)
print("training windows:", len(W_IDS))

training windows: 1067


## 8. Model: Phase-1 backbone (causal) + new bidirectional head

In [8]:
inv_freq = 1 / (10000 ** (np.arange(0, HD, 2) / HD))
angles = np.arange(1280)[:, None] * inv_freq
COS = jnp.asarray(np.cos(angles), jnp.float32)
SIN = jnp.asarray(np.sin(angles), jnp.float32)


def rmsn(x, w):
    return x * lax.rsqrt((x * x).mean(-1, keepdims=True) + 1e-6) * w


def rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2], x[..., HD // 2:]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)


def block(p, x, window, cos, sin, causal, valid=None):
    Bsz, T, _ = x.shape
    h = rmsn(x, p["n1"])
    q = rope(rmsn((h @ p["wq"]).reshape(Bsz, T, H, HD), p["qn"]), cos, sin)
    k = rope(rmsn((h @ p["wk"]).reshape(Bsz, T, KV, HD), p["kn"]), cos, sin)
    v = (h @ p["wv"]).reshape(Bsz, T, KV, HD)

    scores = jnp.einsum("btgrd,bsgd->bgrts", q.reshape(Bsz, T, KV, R, HD), k) * HD ** -0.5
    dist = jnp.arange(T)[:, None] - jnp.arange(T)[None]
    mask = ((dist >= 0) & (dist < window)) if causal else (jnp.abs(dist) < window)
    mask = mask[None] & (valid[:, None, :] if valid is not None else True)

    attn = jax.nn.softmax(jnp.where(mask[:, None, None], scores, -1e30), -1)
    out = jnp.einsum("bgrts,bsgd->btgrd", attn, v).reshape(Bsz, T, H * HD) @ p["wo"]

    g1, g2 = (p["g"][0], p["g"][1]) if "g" in p else (1.0, 1.0)
    x = x + g1 * out
    h = rmsn(x, p["n2"])
    return x + g2 * ((jax.nn.silu(h @ p["w1"]) * (h @ p["w3"])) @ p["w2"])


def logits_fn(P, win, ids, valid):
    T = ids.shape[1]
    cos, sin = COS[:T][None, :, None, :], SIN[:T][None, :, None, :]

    def layer_step(x, z):
        return block(z[0], x, z[1], cos, sin, True), None

    x, _ = lax.scan(layer_step, P["emb"][ids], (P["layers"], win))
    x = rmsn(x, P["nf"])
    x = block(P["head"], x, BIG, cos, sin, False, valid)
    return rmsn(x, P["nh"]) @ P["cls"]


def init_head(key):
    ks = jax.random.split(key, 8)
    n = lambda i, shape: jax.random.normal(ks[i], shape) * 0.02
    ff = cfg["ff"]
    return {
        "wq": n(0, (D, H * HD)), "wk": n(1, (D, KV * HD)), "wv": n(2, (D, KV * HD)), "wo": n(3, (H * HD, D)),
        "n1": jnp.ones(D), "n2": jnp.ones(D), "qn": jnp.ones(HD), "kn": jnp.ones(HD),
        "w1": n(4, (D, ff)), "w3": n(5, (D, ff)), "w2": n(6, (ff, D)),
    }


def build_model(layers, key):
    return {
        "emb": P0["emb"],
        "layers": layers,
        "nf": P0["nf"],
        "head": init_head(key),
        "nh": jnp.ones(D),
        "cls": jax.random.normal(jax.random.fold_in(key, 1), (D, K2)) * 0.02,
    }


def count_params(P):
    return sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(P))


log_probs_fn = jax.jit(lambda P, win, ids, v: jax.nn.log_softmax(logits_fn(P, win, ids, v)))
logits_jit = jax.jit(logits_fn)

## 9. Span decoding, prediction over long documents, strict Macro-F1

In [9]:
def decode_spans(tags, A, B):
    spans, cur = [], None
    for i, t in enumerate(tags):
        if B[i] == A[i]:
            continue
        k = (t + 1) // 2
        if t == 0:
            if cur:
                spans.append(tuple(cur))
                cur = None
        elif t % 2 == 1 or cur is None or cur[2] != k:
            if cur:
                spans.append(tuple(cur))
            cur = [A[i], B[i], k]
        else:
            cur[1] = B[i]
    if cur:
        spans.append(tuple(cur))
    return spans


def predict(P, win, docs):
    out = {}
    for d, (ids, A, B) in docs.items():
        n = len(ids)
        score = np.zeros((n, K2), np.float32)
        count = np.zeros(n)
        starts = window_starts(n, WINDOW, INFER_STRIDE)
        for i in range(0, len(starts), 32):
            chunk = starts[i:i + 32]
            X = np.zeros((32, WINDOW), np.int32)
            Vd = np.zeros((32, WINDOW), bool)
            for j, s in enumerate(chunk):
                seg = ids[s:s + WINDOW]
                X[j, :len(seg)] = seg
                Vd[j, :len(seg)] = True
            lp = np.asarray(log_probs_fn(P, win, X, Vd))
            for j, s in enumerate(chunk):
                m = min(WINDOW, n - s)
                score[s:s + m] += lp[j, :m]
                count[s:s + m] += 1
        out[d] = decode_spans((score / count[:, None]).argmax(1), A, B)
    return out


def strict_scores(pred, gold):
    precs, recs, f1s = [], [], []
    for k in range(1, K + 1):
        tp = fp = fn = 0
        for d in gold:
            g = {(int(s), int(e)) for s, e, l in gold[d] if l == k}
            p = {(int(s), int(e)) for s, e, l in pred.get(d, []) if l == k}
            tp += len(g & p)
            fp += len(p - g)
            fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        precs.append(pr)
        recs.append(rc)
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f1s)), float(np.mean(precs)), float(np.mean(recs)), f1s

## 10. Training loop (AdamW, class-weighted CE, optional distillation loss, time-budgeted)

In [10]:
def make_step(lr_mult, decay_mask):
    def loss_fn(P, win, ids, valid, y, teacher_logits, alpha):
        logits = logits_fn(P, win, ids, valid)
        logp = jax.nn.log_softmax(logits)
        keep = y >= 0
        yy = jnp.maximum(y, 0)
        w = jnp.where(yy == 0, O_WEIGHT, 1.0) * keep
        ce = (-jnp.take_along_axis(logp, yy[..., None], -1)[..., 0] * w).sum() / w.sum()

        t_logp = jax.nn.log_softmax(teacher_logits / KD_TEMP)
        s_logp = jax.nn.log_softmax(logits / KD_TEMP)
        kd = ((jnp.exp(t_logp) * (t_logp - s_logp)).sum(-1) * keep).sum() / keep.sum() * KD_TEMP ** 2
        return (1 - alpha) * ce + alpha * kd

    @jax.jit
    def step(P, m, v, t, lr, win, ids, valid, y, teacher_logits, alpha):
        loss, g = jax.value_and_grad(loss_fn)(P, win, ids, valid, y, teacher_logits, alpha)
        gnorm = jnp.sqrt(sum((x ** 2).sum() for x in jax.tree_util.tree_leaves(g)))
        g = tmap(lambda x: x * jnp.minimum(1.0, 1.0 / (gnorm + 1e-6)), g)
        m = tmap(lambda a, b: 0.9 * a + 0.1 * b, m, g)
        v = tmap(lambda a, b: 0.999 * a + 0.001 * b * b, v, g)

        def update(p, a, b, mult, dec):
            adam = (a / (1 - 0.9 ** t)) / (jnp.sqrt(b / (1 - 0.999 ** t)) + 1e-8)
            return p - lr * mult * (adam + 0.01 * dec * p)

        P = tmap(update, P, m, v, lr_mult, decay_mask)
        return P, m, v, loss

    return step


def train(name, P, win, minutes, teacher_logits=None, alpha=0.0):
    lr_mult = jax.tree_util.tree_map_with_path(
        lambda path, x: HEAD_LR_MULT if path[0].key in ("head", "cls", "nh") else 1.0, P)
    decay_mask = tmap(lambda x: float(x.ndim >= 3), P)
    step = make_step(lr_mult, decay_mask)
    m = tmap(jnp.zeros_like, P)
    v = tmap(jnp.zeros_like, P)
    zeros = jnp.zeros((BATCH, WINDOW, K2))
    steps_per_epoch = len(W_IDS) // BATCH
    seconds = budget(minutes)

    t, total, done = 0, None, False
    best = (-1, None, None)
    t_start = time.time()

    while not done:
        perm = np.random.permutation(len(W_IDS))
        for i in range(steps_per_epoch):
            ix = perm[i * BATCH:(i + 1) * BATCH]
            t += 1
            lr = PEAK_LR * min(1, t / WARMUP_STEPS)
            if total:
                lr *= 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, t / total)))
            tl = jnp.asarray(teacher_logits[ix], jnp.float32) if teacher_logits is not None else zeros
            P, m, v, loss = step(P, m, v, jnp.float32(t), lr, win,
                                 W_IDS[ix], W_VALID[ix], W_Y[ix], tl, jnp.float32(alpha))
            if t == 5:
                loss.block_until_ready()
                t_bench = time.time()
            if t == 25:
                loss.block_until_ready()
                sec_per_step = (time.time() - t_bench) / 20
                total = min(MAX_EPOCHS * steps_per_epoch, int(0.9 * seconds / sec_per_step))
                print(f"[{name}] {sec_per_step:.2f}s/step -> {total} steps ({total / steps_per_epoch:.1f} epochs)")
            if (total and t >= total) or time.time() - T0 > MAX_SECONDS:
                done = True
                break

        scores = strict_scores(predict(P, win, VAL_DOCS), VAL_GOLD)
        print(f"[{name}] step {t} loss {float(loss):.4f} | val strict macro-F1 {scores[0]:.4f} "
              f"P {scores[1]:.4f} R {scores[2]:.4f} | {(time.time() - t_start) / 60:.1f} min", flush=True)
        if scores[0] > best[0]:
            best = (scores[0], P, scores)

    return best, time.time() - t_start


def save_submission(name, P, win):
    t_start = time.time()
    pred = predict(P, win, TEST)
    elapsed = time.time() - t_start
    out_dir = f"/kaggle/working/{name}"
    os.makedirs(out_dir, exist_ok=True)
    result = {d: [[int(s), int(e) - INC, LAB[k - 1]] for s, e, k in spans] for d, spans in pred.items()}
    json.dump(result, open(f"{out_dir}/prediction.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{out_dir}/prediction.json", "prediction.json")
    print(f"{name}: {sum(len(v) for v in result.values())} entities in {len(result)} docs, inference {elapsed:.0f}s")
    return elapsed

## 11. Experiment 1 — Full fine-tuning (also the distillation teacher)

In [11]:
RESULTS = []
WIN_FULL = jnp.asarray(WIN)

model_full = build_model(P0["layers"], jax.random.PRNGKey(0))
(best_f1, teacher, teacher_scores), train_secs = train("full_ft", model_full, WIN_FULL, FULL_FT_MINUTES)
infer_secs = save_submission("p2_full_ft", teacher, WIN_FULL)
RESULTS.append(("full_ft", count_params(teacher), teacher_scores, train_secs, infer_secs))

[full_ft] 0.80s/step -> 1064 steps (8.0 epochs)
[full_ft] step 133 loss 0.1223 | val strict macro-F1 0.3571 P 0.3021 R 0.4616 | 2.3 min
[full_ft] step 266 loss 0.0513 | val strict macro-F1 0.4061 P 0.3633 R 0.4909 | 4.4 min
[full_ft] step 399 loss 0.0285 | val strict macro-F1 0.3809 P 0.3549 R 0.4367 | 6.5 min
[full_ft] step 532 loss 0.0194 | val strict macro-F1 0.3860 P 0.3673 R 0.4417 | 8.6 min
[full_ft] step 665 loss 0.0032 | val strict macro-F1 0.4224 P 0.3920 R 0.4900 | 10.7 min
[full_ft] step 798 loss 0.0037 | val strict macro-F1 0.3946 P 0.3739 R 0.4549 | 12.8 min
[full_ft] step 931 loss 0.0002 | val strict macro-F1 0.3805 P 0.3635 R 0.4328 | 15.0 min
[full_ft] step 1064 loss 0.0004 | val strict macro-F1 0.3818 P 0.3668 R 0.4314 | 17.1 min
p2_full_ft: 5349 entities in 35 docs, inference 42s


## 12. Experiment 2 — Knowledge distillation (6-layer student initialised from Phase-1, teacher = Experiment 1)

In [12]:
student_idx = np.arange(0, L, 2)
WIN_STUDENT = jnp.asarray(WIN[student_idx])

teacher_logits = np.concatenate([
    np.asarray(logits_jit(teacher, WIN_FULL, W_IDS[i:i + 32], W_VALID[i:i + 32]))
    for i in range(0, len(W_IDS), 32)
]).astype(np.float16)
print("teacher logits:", teacher_logits.shape)

student_layers = tmap(lambda a: a[student_idx], P0["layers"])
model_student = build_model(student_layers, jax.random.PRNGKey(1))
(best_f1_s, student, student_scores), train_secs_s = train(
    "kd_student", model_student, WIN_STUDENT, KD_MINUTES, teacher_logits=teacher_logits, alpha=KD_ALPHA)
infer_secs_s = save_submission("p2_kd", student, WIN_STUDENT)
RESULTS.append(("kd_student", count_params(student), student_scores, train_secs_s, infer_secs_s))

teacher logits: (1067, 512, 25)
[kd_student] 0.46s/step -> 1064 steps (8.0 epochs)
[kd_student] step 133 loss 0.5025 | val strict macro-F1 0.2063 P 0.1750 R 0.2724 | 1.3 min
[kd_student] step 266 loss 0.1143 | val strict macro-F1 0.2828 P 0.2804 R 0.3369 | 2.4 min
[kd_student] step 399 loss 0.0591 | val strict macro-F1 0.2986 P 0.2924 R 0.3371 | 3.5 min
[kd_student] step 532 loss 0.0265 | val strict macro-F1 0.3254 P 0.3206 R 0.3617 | 4.7 min
[kd_student] step 665 loss 0.0183 | val strict macro-F1 0.3394 P 0.3291 R 0.3880 | 5.8 min
[kd_student] step 798 loss 0.0073 | val strict macro-F1 0.3623 P 0.3482 R 0.4091 | 6.9 min
[kd_student] step 931 loss 0.0105 | val strict macro-F1 0.3636 P 0.3495 R 0.4131 | 8.1 min
[kd_student] step 1064 loss 0.0085 | val strict macro-F1 0.3654 P 0.3501 R 0.4140 | 9.2 min
p2_kd: 5108 entities in 35 docs, inference 22s


## 13. Results summary

In [13]:
print(f"{'experiment':12s}{'params(M)':>10s}{'macroF1':>9s}{'macroP':>8s}{'macroR':>8s}{'train(min)':>11s}{'test-infer(s)':>14s}")
for name, n_params, sc, tr, inf in RESULTS:
    print(f"{name:12s}{n_params / 1e6:10.1f}{sc[0]:9.4f}{sc[1]:8.4f}{sc[2]:8.4f}{tr / 60:11.1f}{inf:14.0f}")

print()
print(f"{'label':10s}" + "".join(f"{r[0]:>12s}" for r in RESULTS))
for k, label in enumerate(LAB):
    print(f"{label:10s}" + "".join(f"{r[2][3][k]:12.4f}" for r in RESULTS))
print(f"\ntotal time {(time.time() - T0) / 3600:.2f} h")

experiment   params(M)  macroF1  macroP  macroR train(min) test-infer(s)
full_ft           94.4   0.4224  0.3920  0.4900       17.1            42
kd_student        56.7   0.3654  0.3501  0.4140        9.2            22

label          full_ft  kd_student
A.COUNSEL       0.2778      0.1892
APP             0.0948      0.0787
AUTH            0.2994      0.2089
CASENO          0.5059      0.5325
COURT           0.6210      0.7476
DATE            0.9765      0.9306
JUDGE           0.5542      0.3659
PREC            0.3540      0.2734
R.COUNSEL       0.3692      0.2778
RESP            0.0870      0.0565
STAT            0.7575      0.5808
WIT             0.1714      0.1429

total time 0.47 h
